<a href="https://colab.research.google.com/github/MohammedAl-Shareef/Quantum-ML-KEM-ML-DSA-SLH-DSA/blob/mlkem-alshareef/ml_kem_implementation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [21]:
# ML-KEM Full Example (Fixed Errors Version)

import random

# Parameters
q = 17
n = 4
k = 2

# --- Basic Polynomial Operations ---

def poly_add(p1, p2):
    return [(a + b) % q for a, b in zip(p1, p2)]

def poly_sub(p1, p2):
    return [(a - b) % q for a, b in zip(p1, p2)]

def poly_mul(p1, p2):
    res = [0] * (2*n - 1)
    for i in range(n):
        for j in range(n):
            res[i + j] += p1[i] * p2[j]
    for i in range(n, 2*n - 1):
        res[i - n] = (res[i - n] - res[i]) % q
    return [c % q for c in res[:n]]

def format_poly(poly):
    terms = []
    for i, coeff in enumerate(poly):
        if coeff != 0:
            if i == 0:
                terms.append(f"{coeff}")
            elif i == 1:
                terms.append(f"{coeff}x")
            else:
                terms.append(f"{coeff}x^{i}")
    return " + ".join(terms) if terms else "0"

def encode_message(m):
    return [(q//2) * bit for bit in m]

def decode_message(poly):
    return [0 if coeff < q//4 or coeff > (3*q)//4 else 1 for coeff in poly]

def transpose_matrix(A):
    return [
        [A[0][0], A[1][0]],
        [A[0][1], A[1][1]]
    ]

# --- Key and Matrix Definitions ---

def define_secret_key():
    return [[0, 1, -1, 1], [0, 1, 0, 1]]

def define_error_vector():
    return [[1, 0, 1, 0], [-1, 1, 0, 0]]

def define_matrix_A():
    return [
        [[6, 16, 6, 13], [8, 5, 0, 0]],
        [[4, 5, 8, 3], [4, 2, 8, 9]]
    ]

def matrix_vector_mul(A, v):
    result = []
    for row in A:
        acc = [0] * n
        for a_poly, v_poly in zip(row, v):
            acc = poly_add(acc, poly_mul(a_poly, v_poly))
        result.append(acc)
    return result

# --- Algorithm Steps ---

keygen_steps = [
    "1. Select secret key s (fixed polynomials).",
    "2. Select error vector e (fixed polynomials).",
    "3. Define public matrix A (fixed polynomials).",
    "4. Compute public key t = A·s + e (mod q).",
    "5. Output (A, t) as public key, s as secret key."
]

encapsulation_steps = [
    "1. Select random message m (bits).",
    "2. Use fixed small vector r and fixed errors e1 and e2.",
    "3. Compute u = Aᵀ·r + e1.",
    "4. Compute v = tᵀ·r + e2 + encode(m).",
    "5. Output (u, v) as ciphertext."
]

decapsulation_steps = [
    "1. Compute w = v - (sᵀ·u) (mod q).",
    "2. Decode each coefficient of w.",
    "3. If near 0 or q -> decode as 0, if near q/2 -> decode as 1.",
    "4. Recover message m."
]

# --- Key Generation ---

def keygen():
    print("\n=== ALGORITHM: Key Generation ===")
    for step in keygen_steps:
        print(step)

    s = define_secret_key()
    e = define_error_vector()
    A = define_matrix_A()

    print("\nSecret key s:")
    for i in range(k):
        print(f"s[{i}] = {format_poly(s[i])} (mod {q})")

    print("\nError vector e:")
    for i in range(k):
        print(f"e[{i}] = {format_poly(e[i])} (mod {q})")

    print("\nPublic matrix A:")
    for i in range(k):
        for j in range(k):
            print(f"A[{i}][{j}] = {format_poly(A[i][j])} (mod {q})")

    As = matrix_vector_mul(A, s)
    t = [poly_add(As[i], e[i]) for i in range(k)]

    print("\nPublic key component t:")
    for i in range(k):
        print(f"t[{i}] = {format_poly(t[i])} (mod {q})")

    return A, t, s

# --- Encapsulation (Encryption) ---

def encapsulate(A, t):
    print("\n=== ALGORITHM: Encapsulation (Encryption) ===")
    for step in encapsulation_steps:
        print(step)

    # Fixed r, e1, e2
    r = [[1, 0, -1, 1], [0, 1, -1, 0]]         # Fixed random vector
    e1 = [[0, 1, 0, -1], [1, 0, 0, 0]]          # Fixed small error vector
    e2 = [0, -1, 1, 0]                          # Fixed small error polynomial

    print("\nFixed random vector r:")
    for i in range(k):
        print(f"r[{i}] = {format_poly(r[i])}")

    print("\nFixed error vector e1:")
    for i in range(k):
        print(f"e1[{i}] = {format_poly(e1[i])}")

    print("\nFixed error polynomial e2:")
    print(f"e2 = {format_poly(e2)}")

    # Random binary message
    m = [1, 0, 1, 0]
    print("\nFixed binary message m:")
    print(f"m = {m}")

    A_T = transpose_matrix(A)

    u = []
    for i in range(k):
        acc = [0] * n
        for j in range(k):
            acc = poly_add(acc, poly_mul(A_T[i][j], r[j]))
        u.append(poly_add(acc, e1[i]))

    v = [0] * n
    for i in range(k):
        v = poly_add(v, poly_mul(t[i], r[i]))
    v = poly_add(v, e2)
    v = poly_add(v, encode_message(m))

    print("\nCiphertext components u:")
    for i in range(k):
        print(f"u[{i}] = {format_poly(u[i])} (mod {q})")

    print("\nCiphertext component v:")
    print(f"v = {format_poly(v)} (mod {q})")

    return u, v, m

# --- Decapsulation (Decryption) ---

def decapsulate(u, v, s):
    print("\n=== ALGORITHM: Decapsulation (Decryption) ===")
    for step in decapsulation_steps:
        print(step)

    w = v.copy()
    for i in range(k):
        correction = poly_mul(s[i], u[i])
        w = poly_sub(w, correction)

    print("\nRecovered intermediate w:")
    print(f"w = {format_poly(w)} (mod {q})")

    recovered_m = decode_message(w)

    print("\nRecovered binary message m:")
    print(f"m = {recovered_m}")

    return recovered_m

# --- Main Program ---

if __name__ == "__main__":
    print("\n=====================================")
    print("         ML-KEM Simulation           ")
    print("=====================================")

    A, t, s = keygen()
    u, v, m_original = encapsulate(A, t)
    m_recovered = decapsulate(u, v, s)

    print("\n=== FINAL RESULT ===")
    print(f"Original message:  {m_original}")
    print(f"Recovered message: {m_recovered}")
    print("Success:", m_original == m_recovered)



         ML-KEM Simulation           

=== ALGORITHM: Key Generation ===
1. Select secret key s (fixed polynomials).
2. Select error vector e (fixed polynomials).
3. Define public matrix A (fixed polynomials).
4. Compute public key t = A·s + e (mod q).
5. Output (A, t) as public key, s as secret key.

Secret key s:
s[0] = 1x + -1x^2 + 1x^3 (mod 17)
s[1] = 1x + 1x^3 (mod 17)

Error vector e:
e[0] = 1 + 1x^2 (mod 17)
e[1] = -1 + 1x (mod 17)

Public matrix A:
A[0][0] = 6 + 16x + 6x^2 + 13x^3 (mod 17)
A[0][1] = 8 + 5x (mod 17)
A[1][0] = 4 + 5x + 8x^2 + 3x^3 (mod 17)
A[1][1] = 4 + 2x + 8x^2 + 9x^3 (mod 17)

Public key component t:
t[0] = 7 + 4x + 3x^2 + 4x^3 (mod 17)
t[1] = 5 + 13x + 8x^2 + 2x^3 (mod 17)

=== ALGORITHM: Encapsulation (Encryption) ===
1. Select random message m (bits).
2. Use fixed small vector r and fixed errors e1 and e2.
3. Compute u = Aᵀ·r + e1.
4. Compute v = tᵀ·r + e2 + encode(m).
5. Output (u, v) as ciphertext.

Fixed random vector r:
r[0] = 1 + -1x^2 + 1x^3
r[1] = 1